# Task 3: Uniform vs Non-uniform Embeddings

Одна интерактивная Plotly-визуализация для 31 временного ряда:

- слева: non-uniform embedding по лагам из `pecora_unf_res.csv`;
- справа: uniform embedding по `tau` и `dimension` из `acf_fnn_emb.csv`;
- dropdown синхронно переключает выбранный ряд на обоих графиках.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook"

DATA_PATH = Path("dataset_clean.csv")
UNIFORM_PATH = Path("acf_fnn_emb.csv")
NONUNIFORM_PATH = Path("pecora_unf_res.csv")

MAX_POINTS = 3000
NORMALIZE_SERIES = True
MARKER_SIZE = 2.6
MARKER_OPACITY = 0.72

In [ ]:
df = pd.read_csv(DATA_PATH)
uniform_params = pd.read_csv(UNIFORM_PATH)
nonuniform_params = pd.read_csv(NONUNIFORM_PATH)

if "TS" in df.columns:
    data_columns = [col for col in df.columns if col != "TS"]
else:
    data_columns = list(df.columns)

uniform_required = {"column", "tau", "dimension"}
nonuniform_required = {"timeseries_name", "target_dim", "pecora_dim", "pecora_lags", "status"}

missing_uniform = uniform_required - set(uniform_params.columns)
missing_nonuniform = nonuniform_required - set(nonuniform_params.columns)

if missing_uniform:
    raise ValueError(f"Missing columns in {UNIFORM_PATH}: {sorted(missing_uniform)}")
if missing_nonuniform:
    raise ValueError(f"Missing columns in {NONUNIFORM_PATH}: {sorted(missing_nonuniform)}")

nonuniform_ok = nonuniform_params.loc[nonuniform_params["status"].eq("ok")].copy()

common_columns = [
    col for col in uniform_params["column"].tolist()
    if col in data_columns and col in set(nonuniform_ok["timeseries_name"])
]

print(f"Loaded data: {df.shape[0]} rows")
print(f"Common series for visualization: {len(common_columns)}")

if len(common_columns) != 31:
    print("Warning: expected 31 common series")

In [ ]:
def prepare_series(df: pd.DataFrame, column: str, normalize: bool = True) -> np.ndarray:
    series = pd.to_numeric(df[column], errors="coerce").dropna().astype(float)

    if normalize:
        std = series.std(ddof=0)
        if std == 0 or pd.isna(std):
            raise ValueError("constant series")
        series = (series - series.mean()) / std

    return series.to_numpy(dtype=float)


def delay_embedding(series: np.ndarray, m: int, tau: int) -> np.ndarray:
    if m < 3:
        raise ValueError("uniform embedding dimension must be >= 3 for 3D plot")
    if tau < 1:
        raise ValueError("tau must be >= 1")

    x = np.asarray(series, dtype=float)
    x = x[~np.isnan(x)]

    n_points = len(x) - (m - 1) * tau
    if n_points <= 0:
        raise ValueError(f"m={m}, tau={tau} are too large for series length {len(x)}")

    return np.column_stack([
        x[i * tau : i * tau + n_points]
        for i in range(m)
    ])


def nonuniform_embedding(series: np.ndarray, lags: list[int]) -> np.ndarray:
    if len(lags) < 3:
        raise ValueError("at least three lags are required for 3D plot")

    x = np.asarray(series, dtype=float)
    x = x[~np.isnan(x)]
    max_lag = max(lags)

    if len(x) <= max_lag:
        raise ValueError(f"lags={lags} are too large for series length {len(x)}")

    return np.column_stack([
        x[lag : len(x) - max_lag + lag]
        for lag in lags
    ])


def sample_for_plot(X: np.ndarray, max_points: int = MAX_POINTS) -> tuple[np.ndarray, np.ndarray]:
    if len(X) > max_points:
        idx = np.linspace(0, len(X) - 1, max_points).astype(int)
        return X[idx], idx
    return X, np.arange(len(X))


def parse_lags(value) -> list[int]:
    return [int(part) for part in str(value).split(";") if part != ""]

In [ ]:
uniform_by_column = uniform_params.set_index("column")
nonuniform_by_column = nonuniform_ok.set_index("timeseries_name")

embeddings = {}
errors = []

for column in common_columns:
    try:
        series = prepare_series(df, column, normalize=NORMALIZE_SERIES)

        uniform_row = uniform_by_column.loc[column]
        uniform_tau = int(uniform_row["tau"])
        uniform_dim = int(uniform_row["dimension"])
        X_uniform = delay_embedding(series, m=uniform_dim, tau=uniform_tau)

        nonuniform_row = nonuniform_by_column.loc[column]
        lags = parse_lags(nonuniform_row["pecora_lags"])
        X_nonuniform = nonuniform_embedding(series, lags=lags)

        embeddings[column] = {
            "uniform": {
                "X": X_uniform,
                "tau": uniform_tau,
                "dimension": uniform_dim,
                "points": X_uniform.shape[0],
            },
            "nonuniform": {
                "X": X_nonuniform,
                "lags": lags,
                "dimension": int(nonuniform_row["pecora_dim"]),
                "points": X_nonuniform.shape[0],
            },
        }
    except Exception as exc:
        errors.append({"column": column, "error": str(exc)})

summary = pd.DataFrame([
    {
        "column": column,
        "uniform_tau": info["uniform"]["tau"],
        "uniform_dim": info["uniform"]["dimension"],
        "nonuniform_dim": info["nonuniform"]["dimension"],
        "nonuniform_lags": ";".join(map(str, info["nonuniform"]["lags"])),
        "uniform_points": info["uniform"]["points"],
        "nonuniform_points": info["nonuniform"]["points"],
    }
    for column, info in embeddings.items()
])

print(f"Built paired embeddings: {len(embeddings)}")
if errors:
    display(pd.DataFrame(errors))

display(summary)

In [ ]:
def make_comparison_figure(embeddings: dict[str, dict]) -> go.Figure:
    if not embeddings:
        raise ValueError("No paired embeddings to plot")

    columns = list(embeddings.keys())

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        horizontal_spacing=0.02,
        subplot_titles=("Non-uniform Pecora", "Uniform ACF + FNN"),
    )

    for index, column in enumerate(columns):
        info = embeddings[column]
        is_visible = index == 0

        X_non, t_non = sample_for_plot(info["nonuniform"]["X"])
        non_lags = info["nonuniform"]["lags"]
        non_labels = [f"x(t + {lag})" for lag in non_lags]

        fig.add_trace(
            go.Scatter3d(
                x=X_non[:, 0],
                y=X_non[:, 1],
                z=X_non[:, 2],
                mode="markers",
                marker=dict(
                    size=MARKER_SIZE,
                    opacity=MARKER_OPACITY,
                    color=t_non,
                    colorscale="Turbo",
                    showscale=True,
                    colorbar=dict(title="t", x=0.47, len=0.72, thickness=14),
                ),
                name=f"non-uniform: {column}",
                visible=is_visible,
                hovertemplate=(
                    f"{column}<br>"
                    f"{non_labels[0]}=%{{x:.4g}}<br>"
                    f"{non_labels[1]}=%{{y:.4g}}<br>"
                    f"{non_labels[2]}=%{{z:.4g}}<br>"
                    "t=%{marker.color}<extra>non-uniform</extra>"
                ),
                legendgroup=column,
                showlegend=False,
            ),
            row=1,
            col=1,
        )

        X_uni, t_uni = sample_for_plot(info["uniform"]["X"])
        tau = info["uniform"]["tau"]

        fig.add_trace(
            go.Scatter3d(
                x=X_uni[:, 0],
                y=X_uni[:, 1],
                z=X_uni[:, 2],
                mode="markers",
                marker=dict(
                    size=MARKER_SIZE,
                    opacity=MARKER_OPACITY,
                    color=t_uni,
                    colorscale="Turbo",
                    showscale=False,
                ),
                name=f"uniform: {column}",
                visible=is_visible,
                hovertemplate=(
                    f"{column}<br>"
                    "x(t)=%{x:.4g}<br>"
                    f"x(t + {tau})=%{{y:.4g}}<br>"
                    f"x(t + {2 * tau})=%{{z:.4g}}<br>"
                    "t=%{marker.color}<extra>uniform</extra>"
                ),
                legendgroup=column,
                showlegend=False,
            ),
            row=1,
            col=2,
        )

    buttons = []
    for index, column in enumerate(columns):
        info = embeddings[column]
        non = info["nonuniform"]
        uni = info["uniform"]
        non_labels = [f"x(t + {lag})" for lag in non["lags"]]

        visible = [False] * (2 * len(columns))
        visible[2 * index] = True
        visible[2 * index + 1] = True

        title = (
            f"{column}: non-uniform lags={non['lags']} | "
            f"uniform m={uni['dimension']}, tau={uni['tau']}"
        )

        buttons.append(
            dict(
                label=column,
                method="update",
                args=[
                    {"visible": visible},
                    {
                        "title": {"text": title, "x": 0.5},
                        "scene": dict(
                            xaxis_title=non_labels[0],
                            yaxis_title=non_labels[1],
                            zaxis_title=non_labels[2],
                            aspectmode="data",
                        ),
                        "scene2": dict(
                            xaxis_title="x(t)",
                            yaxis_title=f"x(t + {uni['tau']})",
                            zaxis_title=f"x(t + {2 * uni['tau']})",
                            aspectmode="data",
                        ),
                    },
                ],
            )
        )

    first_column = columns[0]
    first = embeddings[first_column]
    first_non = first["nonuniform"]
    first_uni = first["uniform"]
    first_non_labels = [f"x(t + {lag})" for lag in first_non["lags"]]

    fig.update_layout(
        title=dict(
            text=(
                f"{first_column}: non-uniform lags={first_non['lags']} | "
                f"uniform m={first_uni['dimension']}, tau={first_uni['tau']}"
            ),
            x=0.5,
        ),
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=0.0,
                y=1.08,
                xanchor="left",
                yanchor="top",
                showactive=True,
                bgcolor="rgba(255,255,255,0.95)",
                bordercolor="rgba(70,70,70,0.25)",
                borderwidth=1,
            )
        ],
        scene=dict(
            xaxis_title=first_non_labels[0],
            yaxis_title=first_non_labels[1],
            zaxis_title=first_non_labels[2],
            aspectmode="data",
            camera=dict(eye=dict(x=1.45, y=1.45, z=1.1)),
        ),
        scene2=dict(
            xaxis_title="x(t)",
            yaxis_title=f"x(t + {first_uni['tau']})",
            zaxis_title=f"x(t + {2 * first_uni['tau']})",
            aspectmode="data",
            camera=dict(eye=dict(x=1.45, y=1.45, z=1.1)),
        ),
        template="plotly_white",
        width=1250,
        height=720,
        margin=dict(l=10, r=10, b=10, t=105),
        font=dict(size=12),
    )

    fig.update_scenes(
        xaxis=dict(backgroundcolor="rgb(248,249,252)", gridcolor="rgb(225,229,235)", zerolinecolor="rgb(210,215,225)"),
        yaxis=dict(backgroundcolor="rgb(248,249,252)", gridcolor="rgb(225,229,235)", zerolinecolor="rgb(210,215,225)"),
        zaxis=dict(backgroundcolor="rgb(248,249,252)", gridcolor="rgb(225,229,235)", zerolinecolor="rgb(210,215,225)"),
    )

    return fig


comparison_fig = make_comparison_figure(embeddings)
comparison_fig.show()